# Intrinsic vs. Extrinsic Safety Constraints for RL Trading Agents
## Full Experiment Pipeline (Colab-Compatible)

This notebook runs the complete experimental pipeline:
1. **Setup** - Install dependencies, configure GPU
2. **Experiments** - 4 algorithms x 3 rewards x 2 gateways x 50 seeds (with checkpointing)
3. **Analysis** - Statistical tests, ANOVA, pairwise comparisons
4. **Figures** - 9 publication-quality figures
5. **Tables** - LaTeX tables for paper
6. **Ablation** - Reward hyperparameter sensitivity study

**Runtime estimate (Colab T4 GPU)**:
- Full experiment (50 seeds, 1M steps): ~18-24 hours
- Reduced experiment (10 seeds, 500K steps): ~2-3 hours
- Ablation study (marginal, 10 seeds): ~3-4 hours

**Checkpointing**: Progress is saved after each seed. If Colab disconnects, re-run the experiment cell and it will resume from the last checkpoint.

---
## 1. Setup

In [ ]:
# @title 1a. Install Dependencies
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Core dependencies
install('gymnasium>=0.29')
install('stable-baselines3[extra]>=2.3')
install('torch>=2.2')  # Should already be on Colab
install('scipy>=1.13')
install('matplotlib>=3.8')
install('seaborn>=0.13')
install('rich>=13.7')
install('joblib>=1.3')

print('All dependencies installed.')

In [ ]:
# @title 1b. Clone/Upload Repository
import os
from pathlib import Path

# Option A: Clone from git (uncomment and set your repo URL)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/res1

# Option B: Upload via Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/res1 /content/res1

# Option C: Already uploaded / mounted
# Set PROJECT_ROOT to wherever the code lives
PROJECT_ROOT = Path('/content/res1')

# For local testing, uncomment:
# PROJECT_ROOT = Path('.').resolve().parent

assert PROJECT_ROOT.exists(), f'Project root not found at {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'agentic_trader').exists(), 'agentic_trader package not found'

# Add to Python path
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Contents: {[p.name for p in PROJECT_ROOT.iterdir() if not p.name.startswith(".")]}')

In [ ]:
# @title 1c. Verify GPU & Environment
import torch
import gymnasium as gym
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

# Quick smoke test
from agentic_trader.env.abides_env import TradingEnv
from agentic_trader.config.settings import EnvConfig

env = TradingEnv(config=EnvConfig(episode_length=10), seed=42)
obs, _ = env.reset()
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, term, trunc, info = env.step(action)
print(f'Environment smoke test passed. Obs shape: {obs.shape}, Final PnL: ${info["pnl"]:+,.2f}')

---
## 2. Configure Experiment

In [ ]:
# @title 2. Experiment Configuration {run: "auto"}

# ============================================================
# MAIN EXPERIMENT SETTINGS — adjust these for your run
# ============================================================

# Full experiment (paper-ready): seeds=50, train_steps=1_000_000
# Reduced (quick validation):   seeds=10, train_steps=500_000
# Smoke test:                    seeds=2,  train_steps=50_000

N_SEEDS = 50                    # @param {type:"integer"}
TRAIN_STEPS = 1_000_000         # @param {type:"integer"}
EVAL_EPISODES = 30              # @param {type:"integer"}
N_ENVS = 4                      # @param {type:"integer"}
ALGORITHMS = ['ppo', 'sac', 'td3', 'lag_ppo']  # all four

# Ablation settings
ABLATION_SEEDS = 10             # @param {type:"integer"}
ABLATION_STEPS = 500_000        # @param {type:"integer"}
ABLATION_SWEEP = 'marginal'     # 'marginal' or 'joint'

# Output directory
OUTPUT_DIR = str(PROJECT_ROOT / 'output' / 'experiments')
ABLATION_DIR = str(PROJECT_ROOT / 'output' / 'ablation')

# Create output dirs
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(ABLATION_DIR).mkdir(parents=True, exist_ok=True)

# Estimate runtime
n_models = len(ALGORITHMS) * 3  # 3 reward types
n_conditions = 2 + len(ALGORITHMS) * 3 * 2  # baselines + factorial
total_episodes = N_SEEDS * n_conditions * EVAL_EPISODES
total_train_runs = N_SEEDS * n_models

print('=' * 60)
print('EXPERIMENT PLAN')
print('=' * 60)
print(f'Algorithms:      {ALGORITHMS}')
print(f'Seeds:           {N_SEEDS}')
print(f'Train steps:     {TRAIN_STEPS:,} per model')
print(f'Training runs:   {total_train_runs:,}')
print(f'Conditions:      {n_conditions}')
print(f'Eval episodes:   {total_episodes:,} total')
print(f'Device:          {DEVICE}')
print(f'Output:          {OUTPUT_DIR}')
print('=' * 60)

---
## 3. Run Main Experiments

This cell runs the full factorial experiment. Progress is checkpointed after each seed.

**If Colab disconnects**: Simply re-run this cell. It will detect the checkpoint and resume.

In [ ]:
# @title 3. Run Experiments (with checkpointing)
import time
import json
from dataclasses import asdict

from scripts.run_experiments import (
    ExperimentConfig,
    EpisodeResult,
    build_conditions,
    process_seed,
    aggregate_results,
    _load_checkpoint,
    _save_checkpoint,
    _resolve_device,
)

cfg = ExperimentConfig(
    n_seeds=N_SEEDS,
    train_steps=TRAIN_STEPS,
    eval_episodes=EVAL_EPISODES,
    n_envs=N_ENVS,
    algorithms=ALGORITHMS,
    n_jobs=1,
    device=DEVICE,
    checkpoint=True,
    resume=True,  # Always try to resume
)
cfg.output_dir = OUTPUT_DIR

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Check for existing checkpoint
existing_results, completed_seeds = _load_checkpoint(out_dir)
remaining = [i for i in range(N_SEEDS) if i not in completed_seeds]

if completed_seeds:
    print(f'Resuming from checkpoint: {len(completed_seeds)}/{N_SEEDS} seeds done')
    print(f'  Cached episodes: {len(existing_results):,}')
    print(f'  Remaining seeds: {len(remaining)}')
else:
    print(f'Starting fresh: {N_SEEDS} seeds to process')

all_results_dicts = list(existing_results)
t_start = time.time()

for idx, seed_idx in enumerate(remaining):
    t_seed = time.time()
    print(f'\n{"="*60}')
    print(f'Seed {seed_idx+1}/{N_SEEDS} (remaining: {len(remaining)-idx-1})')
    print(f'{"="*60}')

    seed_results = process_seed(seed_idx, cfg)
    new_dicts = [asdict(r) for r in seed_results]
    all_results_dicts.extend(new_dicts)
    completed_seeds.add(seed_idx)

    _save_checkpoint(out_dir, all_results_dicts, completed_seeds)

    elapsed_seed = time.time() - t_seed
    elapsed_total = time.time() - t_start
    seeds_done = len(completed_seeds)
    seeds_left = N_SEEDS - seeds_done
    avg_per_seed = elapsed_total / max(idx + 1, 1)
    eta = avg_per_seed * seeds_left

    print(f'  Seed done in {elapsed_seed:.0f}s | '
          f'Progress: {seeds_done}/{N_SEEDS} | '
          f'ETA: {eta/3600:.1f}h')

# Save final results
raw_path = out_dir / 'raw_results.json'
raw_path.write_text(json.dumps(all_results_dicts, indent=2))

# Build summary
all_episodes = []
for d in all_results_dicts:
    all_episodes.append(EpisodeResult(**{k: v for k, v in d.items() if k in EpisodeResult.__dataclass_fields__}))
summary = aggregate_results(all_episodes)
summary_path = out_dir / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

# Clean checkpoint
ckpt = out_dir / 'checkpoint_results.json'
if ckpt.exists():
    ckpt.unlink()

total_time = time.time() - t_start
print(f'\n{"="*60}')
print(f'EXPERIMENTS COMPLETE')
print(f'  Total time: {total_time/3600:.1f} hours')
print(f'  Episodes: {len(all_results_dicts):,}')
print(f'  Results: {raw_path}')
print(f'{"="*60}')

---
## 4. Statistical Analysis

In [ ]:
# @title 4. Run Statistical Analysis
from scripts.analyze_results import load_results, generate_report

raw_path = str(Path(OUTPUT_DIR) / 'raw_results.json')
results = load_results(raw_path)
print(f'Loaded {len(results):,} episode results.')

report = generate_report(results)

report_path = Path(OUTPUT_DIR) / 'analysis_report.md'
report_path.write_text(report)
print(f'Report saved to {report_path}')

# Display in notebook
from IPython.display import Markdown, display
display(Markdown(report))

---
## 5. Generate Figures

In [ ]:
# @title 5. Generate Publication Figures
import matplotlib
matplotlib.use('Agg')

from scripts.generate_figures import (
    load_results, detect_algorithms,
    fig1_main_results, fig2_safety, fig3_risk_return,
    fig4_interaction, fig5_boxplots, fig6_reduction,
    fig7_cross_algorithm, fig8_passivity, fig9_lagrangian,
)

raw_path = str(Path(OUTPUT_DIR) / 'raw_results.json')
results = load_results(raw_path)
algorithms = detect_algorithms(results)
print(f'Detected algorithms: {algorithms}')

fig_dir = Path(OUTPUT_DIR) / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

fig1_main_results(results, fig_dir, algorithms)
fig2_safety(results, fig_dir, algorithms)
fig3_risk_return(results, fig_dir, algorithms)
fig4_interaction(results, fig_dir, algorithms)
fig5_boxplots(results, fig_dir, algorithms)
fig6_reduction(results, fig_dir, algorithms)
fig7_cross_algorithm(results, fig_dir, algorithms)
fig8_passivity(results, fig_dir, algorithms)
fig9_lagrangian(results, fig_dir, algorithms)

print(f'\nAll 9 figures saved to {fig_dir}/')
print('Files:', [f.name for f in sorted(fig_dir.glob('*.png'))])

In [ ]:
# @title 5b. Display Key Figures Inline
from IPython.display import Image, display

fig_dir = Path(OUTPUT_DIR) / 'figures'

key_figs = ['fig1_main_results.png', 'fig2_safety.png', 'fig7_cross_algorithm.png',
            'fig8_passivity.png', 'fig9_lagrangian.png']

for name in key_figs:
    path = fig_dir / name
    if path.exists():
        print(f'\n--- {name} ---')
        display(Image(filename=str(path), width=800))

---
## 6. Generate LaTeX Tables

In [ ]:
# @title 6. Generate LaTeX Tables for Paper
from scripts.generate_latex_tables import (
    load_results, detect_algorithms,
    table_main, table_cross_algorithm, table_passivity, table_lagrangian,
)

raw_path = str(Path(OUTPUT_DIR) / 'raw_results.json')
results = load_results(raw_path)
algorithms = detect_algorithms(results)

paper_dir = PROJECT_ROOT / 'paper'
paper_dir.mkdir(parents=True, exist_ok=True)

tables = []

# Per-algorithm results tables
for algo in algorithms:
    t = table_main(results, algo)
    tables.append(t)
    (paper_dir / f'results_table_{algo}.tex').write_text(t)
    print(f'Table: results_table_{algo}.tex')

# Cross-algorithm consistency
t2 = table_cross_algorithm(results, algorithms)
tables.append(t2)
(paper_dir / 'cross_algorithm_table.tex').write_text(t2)
print('Table: cross_algorithm_table.tex')

# Passivity analysis
t3 = table_passivity(results, algorithms)
tables.append(t3)
(paper_dir / 'passivity_table.tex').write_text(t3)
print('Table: passivity_table.tex')

# Lagrangian comparison
if 'lag_ppo' in algorithms:
    t4 = table_lagrangian(results)
    tables.append(t4)
    (paper_dir / 'lagrangian_table.tex').write_text(t4)
    print('Table: lagrangian_table.tex')

# Combined
combined = '\n\n% ' + '=' * 70 + '\n\n'.join(tables)
(paper_dir / 'paper_tables.tex').write_text(combined)
print(f'\nAll {len(tables)} tables saved to {paper_dir}/')

---
## 7. Ablation Study

Sweeps reward shaping hyperparameters to demonstrate robustness.
Addresses reviewer concern about coefficients being "tuned to win".

In [ ]:
# @title 7a. Run Ablation Study
from scripts.run_ablation import (
    AblationConfig,
    INV_LAMBDAS, DD_LAMBDAS, VAR_LAMBDAS,
    run_ablation_point, aggregate_ablation,
)
from scripts.run_experiments import PassthroughGateway
import time

acfg = AblationConfig(
    n_seeds=ABLATION_SEEDS,
    train_steps=ABLATION_STEPS,
    eval_episodes=20,
    algorithm='ppo',
    device=DEVICE,
)
acfg.output_dir = ABLATION_DIR

abl_dir = Path(ABLATION_DIR)
abl_dir.mkdir(parents=True, exist_ok=True)

# Build sweep points (marginal sweeps)
base_inv, base_dd, base_var = 1e-3, 5e-3, 1e-3
sweep_points = []
for lam in INV_LAMBDAS:
    sweep_points.append((lam, base_dd, base_var, 'inv'))
for lam in DD_LAMBDAS:
    sweep_points.append((base_inv, lam, base_var, 'dd'))
for lam in VAR_LAMBDAS:
    sweep_points.append((base_inv, base_dd, lam, 'var'))
sweep_points = list(dict.fromkeys(sweep_points))

total_runs = len(sweep_points) * ABLATION_SEEDS
print(f'Ablation: {len(sweep_points)} sweep points x {ABLATION_SEEDS} seeds = {total_runs} runs')

# Load checkpoint
ckpt_path = abl_dir / 'ablation_checkpoint.json'
all_abl_results = []
completed = set()
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text())
    all_abl_results = ckpt.get('results', [])
    completed = set(ckpt.get('completed', []))
    print(f'Resuming: {len(completed)} runs completed')

t_start = time.time()
run_idx = 0

for inv_l, dd_l, var_l, sweep_type in sweep_points:
    for seed_idx in range(ABLATION_SEEDS):
        run_key = f'{inv_l}_{dd_l}_{var_l}_{seed_idx}'
        if run_key in completed:
            run_idx += 1
            continue

        seed = seed_idx * 7 + 42
        run_idx += 1
        print(f'[{run_idx}/{total_runs}] {sweep_type}: '
              f'inv={inv_l:.4f} dd={dd_l:.4f} var={var_l:.4f} seed={seed}')

        episodes = run_ablation_point(seed, inv_l, dd_l, var_l, acfg)
        agg = aggregate_ablation(episodes, seed, inv_l, dd_l, var_l, sweep_type)
        all_abl_results.append(asdict(agg))
        completed.add(run_key)

        ckpt_path.write_text(json.dumps({
            'completed': sorted(completed),
            'results': all_abl_results,
        }))

# Save final
(abl_dir / 'ablation_results.json').write_text(json.dumps(all_abl_results, indent=2))
if ckpt_path.exists():
    ckpt_path.unlink()

print(f'\nAblation complete in {(time.time()-t_start)/3600:.1f}h ({len(all_abl_results)} results)')

In [ ]:
# @title 7b. Visualize Ablation Results
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

abl_path = Path(ABLATION_DIR) / 'ablation_results.json'
if not abl_path.exists():
    print('No ablation results found. Run cell 7a first.')
else:
    abl_data = json.loads(abl_path.read_text())
    print(f'Loaded {len(abl_data)} ablation results')

    # Group by sweep type
    by_sweep = defaultdict(list)
    for r in abl_data:
        by_sweep[r['sweep_type']].append(r)

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('Ablation Study: Reward Shaping Hyperparameter Sensitivity',
                 fontsize=14, fontweight='bold')

    metrics = [
        ('mean_violations', 'Safety Violations'),
        ('mean_sharpe', 'Sharpe Ratio'),
        ('mean_max_drawdown', 'Max Drawdown ($)'),
    ]
    sweep_configs = [
        ('inv', 'inv_lambda', r'$\lambda_{inv}$', INV_LAMBDAS),
        ('dd', 'dd_lambda', r'$\lambda_{dd}$', DD_LAMBDAS),
        ('var', 'var_lambda', r'$\lambda_{VaR}$', VAR_LAMBDAS),
    ]

    for row, (metric_key, metric_label) in enumerate(metrics):
        for col, (sweep_type, param_key, param_label, param_vals) in enumerate(sweep_configs):
            ax = axes[row, col]
            runs = by_sweep.get(sweep_type, [])
            if not runs:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center')
                continue

            # Group by parameter value
            by_val = defaultdict(list)
            for r in runs:
                by_val[r[param_key]].append(r[metric_key])

            x_vals = sorted(by_val.keys())
            means = [np.mean(by_val[v]) for v in x_vals]
            stds = [np.std(by_val[v]) for v in x_vals]

            ax.errorbar(range(len(x_vals)), means, yerr=stds,
                       marker='o', capsize=4, linewidth=2)
            ax.set_xticks(range(len(x_vals)))
            ax.set_xticklabels([f'{v:.4f}' for v in x_vals], rotation=45, fontsize=7)
            ax.set_xlabel(param_label)
            if col == 0:
                ax.set_ylabel(metric_label)
            if row == 0:
                ax.set_title(f'Sweep: {param_label}')

            # Highlight baseline
            baseline_map = {'inv': 1e-3, 'dd': 5e-3, 'var': 1e-3}
            base = baseline_map[sweep_type]
            if base in x_vals:
                idx = x_vals.index(base)
                ax.axvline(idx, color='red', linestyle='--', alpha=0.5, label='Baseline')

    plt.tight_layout()
    fig_path = Path(ABLATION_DIR) / 'ablation_sensitivity.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    fig.savefig(str(fig_path).replace('.png', '.pdf'), bbox_inches='tight')
    plt.show()
    print(f'Ablation figure saved to {fig_path}')

---
## 8. Quick Sanity Checks (Audit)

In [ ]:
# @title 8. Audit Results
from collections import defaultdict
import json
import numpy as np
from scipy import stats as scipy_stats

raw_path = Path(OUTPUT_DIR) / 'raw_results.json'
results = json.loads(raw_path.read_text())
print(f'Auditing {len(results):,} episodes\n')

# Group by condition
by_cond = defaultdict(list)
for r in results:
    by_cond[r['condition']].append(r)

# 1. Seed coverage
print('=== Seed Coverage ===')
for cond, eps in sorted(by_cond.items()):
    seeds = set(e['seed'] for e in eps)
    print(f'  {cond:<25} {len(seeds):>3} seeds, {len(eps):>5} episodes')

# 2. Risk-adjusted violation check
print('\n=== Safety Violation Summary ===')
for cond in sorted(by_cond):
    eps = by_cond[cond]
    viols = [e.get('safety_violations', 0) for e in eps]
    mean_v = np.mean(viols)
    flag = ' <-- ZERO' if mean_v == 0 else (' *** HIGH' if mean_v > 50 else '')
    print(f'  {cond:<25} mean={mean_v:>7.1f}  max={max(viols):>5}{flag}')

# 3. Key claim verification
print('\n=== Key Claims Verification ===')
algos = sorted(set(r.get('algorithm', 'n/a') for r in results if r.get('algorithm', 'n/a') != 'n/a'))
for algo in algos:
    std_cond = f'{algo}_std_naked'
    risk_cond = f'{algo}_risk_naked'
    if std_cond not in by_cond or risk_cond not in by_cond:
        continue
    std_viols = [e['safety_violations'] for e in by_cond[std_cond]]
    risk_viols = [e['safety_violations'] for e in by_cond[risk_cond]]
    t, p = scipy_stats.ttest_ind(std_viols, risk_viols, equal_var=False)
    print(f'  {algo.upper()}: Std violations={np.mean(std_viols):.1f} vs '
          f'Risk violations={np.mean(risk_viols):.1f}  (p={p:.6f})')

# 4. Gateway redundancy check
print('\n=== Gateway Redundancy (Risk-Adj) ===')
for algo in algos:
    naked = f'{algo}_risk_naked'
    gw = f'{algo}_risk_riskgw'
    if naked not in by_cond or gw not in by_cond:
        continue
    v_naked = np.mean([e['safety_violations'] for e in by_cond[naked]])
    v_gw = np.mean([e['safety_violations'] for e in by_cond[gw]])
    rej = np.mean([e.get('n_rejected', 0) / max(e.get('n_orders', 1), 1) for e in by_cond[gw]])
    print(f'  {algo.upper()}: naked={v_naked:.1f} vs GW={v_gw:.1f}, rej_rate={rej:.2%}')

print('\nAudit complete.')

---
## 9. Download Results

In [ ]:
# @title 9. Package & Download Results
import shutil

# Create a zip of all outputs
output_root = PROJECT_ROOT / 'output'
zip_path = PROJECT_ROOT / 'experiment_results'
shutil.make_archive(str(zip_path), 'zip', str(output_root))
print(f'Results packaged: {zip_path}.zip')
print(f'Size: {(zip_path.with_suffix(".zip")).stat().st_size / 1e6:.1f} MB')

# If on Colab, offer download
try:
    from google.colab import files
    files.download(str(zip_path) + '.zip')
except ImportError:
    print('Not on Colab - find the zip at:', str(zip_path) + '.zip')

# Also save to Drive if mounted
drive_path = Path('/content/drive/MyDrive/experiment_results')
if drive_path.parent.exists():
    drive_path.mkdir(exist_ok=True)
    shutil.copy2(str(zip_path) + '.zip', str(drive_path / 'experiment_results.zip'))
    print(f'Also saved to Google Drive: {drive_path}')

---
## Summary

After running all cells, you should have:

| Output | Location |
|--------|----------|
| Raw episode data | `output/experiments/raw_results.json` |
| Summary statistics | `output/experiments/summary.json` |
| Statistical report | `output/experiments/analysis_report.md` |
| 9 publication figures | `output/experiments/figures/fig*.png` |
| LaTeX tables | `paper/*.tex` |
| Ablation results | `output/ablation/ablation_results.json` |
| Ablation figure | `output/ablation/ablation_sensitivity.png` |

### Experimental Design
- **4 algorithms**: PPO, SAC, TD3, Lagrangian PPO
- **3 reward functions**: Standard, Risk-Adjusted, Sharpe-Inspired
- **2 gateway conditions**: With/Without deterministic risk gateway
- **2 baselines**: Random policy, Mean-Reversion Heuristic
- **50 seeds** x **30 eval episodes** = 1,500 episodes per condition
- **Ablation**: Marginal sweeps on 3 hyperparameters x 4 values x 10 seeds